# 022 — Multi-Query RAG
**Advanced RAG Series** | Generate multiple queries to improve recall

Covers: Query expansion · LangChain MultiQueryRetriever · RAG-Fusion · RRF merge


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile


In [3]:
import os
DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(f"Missing: {path}\nRun python create_datasets.py from project root.")

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, rag_text])
print(f"✓ Loaded {len(all_text):,} chars across 4 files")


✓ Loaded 22,486 chars across 4 files


In [4]:
# ── 1. Build a vector store for retrieval ───────────────────────────────
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
docs_lc = splitter.create_documents([rag_text, ml_text])

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
vectorstore = FAISS.from_documents(docs_lc, embeddings)
print(f"✓ Vector store: {vectorstore.index.ntotal} vectors")


✓ Vector store: 42 vectors


In [5]:
# ── 2. Problem with single-query retrieval ──────────────────────────────
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

query = "How do transformer models handle context?"
results = retriever.invoke(query)
print(f"Query: {query!r}")
print(f"Retrieved {len(results)} chunks")
print("\nTop result:")
print(results[0].page_content[:300])
print()
print("Problem: One query = one perspective. Misses chunks that use")
print("different vocabulary for the same concept.")


Query: 'How do transformer models handle context?'
Retrieved 4 chunks

Top result:
3. Generation: The retrieved chunks are added to the prompt as context, and the language model
   generates a response based on both the query and the retrieved context.

Document Chunking Strategies

Problem: One query = one perspective. Misses chunks that use
different vocabulary for the same concept.


In [6]:
# ── 3. Generate multiple query variants with LLM ────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

QUERY_GEN_PROMPT = ChatPromptTemplate.from_template(
    "You are an AI assistant. Generate 4 different versions of the question below.\n"
    "Each version should use different vocabulary and phrasing but ask the same thing.\n"
    "Output ONLY the 4 questions, one per line, no numbering, no extra text.\n\n"
    "Original question: {question}"
)

query_gen_chain = QUERY_GEN_PROMPT | llm | StrOutputParser()

original_query = "How do transformer models handle context?"
generated = query_gen_chain.invoke({"question": original_query})
variants = [q.strip() for q in generated.strip().split("\n") if q.strip()]

print(f"Original: {original_query}")
print(f"\nGenerated {len(variants)} variants:")
for i, v in enumerate(variants, 1):
    print(f"  {i}. {v}")


Original: How do transformer models handle context?

Generated 4 variants:
  1. What mechanisms do transformer models employ to process contextual information?
  2. How do transformer models incorporate contextual dependencies into their decision-making processes?
  3. In what ways do transformer models utilize contextual data to inform their predictions and outputs?
  4. What strategies do transformer models use to capture and utilize contextual relationships in language?


In [7]:
# ── 4. Retrieve for each query variant ──────────────────────────────────
all_queries = [original_query] + variants
all_retrieved = {}

for q in all_queries:
    docs = retriever.invoke(q)
    for doc in docs:
        key = doc.page_content[:100]  # deduplicate by content prefix
        if key not in all_retrieved:
            all_retrieved[key] = doc

unique_docs = list(all_retrieved.values())
print(f"Single query retrieved: {len(results)} chunks")
print(f"Multi-query retrieved : {len(unique_docs)} unique chunks (after dedup)")
print(f"Recall gain           : +{len(unique_docs) - len(results)} extra chunks")


Single query retrieved: 4 chunks
Multi-query retrieved : 8 unique chunks (after dedup)
Recall gain           : +4 extra chunks


In [8]:
# ── 5. RRF (Reciprocal Rank Fusion) merge ───────────────────────────────
# RRF re-ranks merged results by combining rank positions across all queries

def reciprocal_rank_fusion(query_results: list[list], k: int = 60) -> list:
    scores = {}
    doc_map = {}
    for results in query_results:
        for rank, doc in enumerate(results):
            key = doc.page_content[:100]
            doc_map[key] = doc
            scores[key] = scores.get(key, 0) + 1 / (k + rank + 1)
    sorted_keys = sorted(scores, key=lambda x: scores[x], reverse=True)
    return [(doc_map[k], scores[k]) for k in sorted_keys]

per_query_results = [retriever.invoke(q) for q in all_queries]
fused = reciprocal_rank_fusion(per_query_results)

print("RRF-fused top 5 results:")
for i, (doc, score) in enumerate(fused[:5], 1):
    print(f"\n[{i}] RRF score={score:.4f}")
    print(doc.page_content[:200])


RRF-fused top 5 results:

[1] RRF score=0.0794
Transfer Learning

Transfer learning is a machine learning technique where a model trained on one task is
reused as the starting point for a model on a second task. Fine-tuning a pretrained model
like

[2] RRF score=0.0650
RAG was introduced by Lewis et al. (2020) as a way to give language models access to
up-to-date information and verifiable sources, addressing key limitations of parametric
(weights-only) language mod

[3] RRF score=0.0648
3. Generation: The retrieved chunks are added to the prompt as context, and the language model
   generates a response based on both the query and the retrieved context.

Document Chunking Strategies

[4] RRF score=0.0471
through unsupervised learning. Some implementations of machine learning use data and neural
networks in a way that mimics the working of a biological brain.

[5] RRF score=0.0164
Generative models: learning the underlying distribution of data to generate new samples.
Variational autoe

In [9]:
# ── 6. Full multi-query RAG answer ──────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer ONLY using the context below.\n"
    "If the answer is not in the context, say \"I don't know\".\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\nAnswer:"
)

def multi_query_rag(question: str, top_k: int = 5) -> str:
    # Step 1: generate query variants
    variants_raw = query_gen_chain.invoke({"question": question})
    all_qs = [question] + [v.strip() for v in variants_raw.strip().split("\n") if v.strip()]

    # Step 2: retrieve for each
    per_q = [retriever.invoke(q) for q in all_qs]

    # Step 3: RRF merge + take top_k
    fused = reciprocal_rank_fusion(per_q)[:top_k]

    # Step 4: generate
    context = "\n\n---\n\n".join([doc.page_content for doc, _ in fused])
    chain = RAG_PROMPT | llm
    return chain.invoke({"context": context, "question": question}).content

answer = multi_query_rag("How do transformer models handle long context?")
print(answer)


I don't know.


In [10]:
# ── 7. LangChain MultiQueryRetriever (built-in) ──────────────────────────
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    llm=llm,
)

docs = multi_retriever.invoke("explain attention mechanism in transformers")
print(f"\nMultiQueryRetriever returned {len(docs)} unique docs")
print("\nTop doc:")
print(docs[0].page_content[:300])



MultiQueryRetriever returned 7 unique docs

Top doc:
1. Ingestion: Documents are loaded, split into chunks, embedded into vectors, and stored in
   a vector database. The chunk size and overlap are critical hyperparameters.

2. Retrieval: Given a user query, the query is embedded and the most similar document chunks
   are retrieved from the vector da
